In [ ]:
# WEBNOVEL TITLE + DESCRIPTION -> GENRE/TAG PREDICTOR

!pip -q install scikit-learn joblib pandas

import pandas as pd
import numpy as np
import re
import joblib

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# csv upload
uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Loaded:", filename)
print("Number of novels:", len(df))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

# keep only title, description and tags
df = df[["title", "description", "tags"]].copy()
df = df.dropna(subset=["title", "description", "tags"])

df["title"] = df["title"].astype(str)
df["description"] = df["description"].astype(str)
df["tags"] = df["tags"].astype(str)

# remove non-english titles & descriptions
def contains_non_english_characters(text):
    for char in str(text):
        if char.isalpha() and not ("a" <= char.lower() <= "z"):
            return True

    return False

non_english_title = df["title"].apply(
    contains_non_english_characters
)

non_english_description = df["description"].apply(
    contains_non_english_characters
)

# keep only rows where both title and description are english
df = df[
    ~non_english_title
    & ~non_english_description
].copy()

print("\nRows after removing non-English titles/descriptions:", len(df))

print("\nRows after removing missing data:", len(df))

def clean_title(title):
    title = title.lower()

    # Remove URLs
    title = re.sub(r"https?://\S+", " ", title)

    # Replace punctuation with spaces
    title = re.sub(r"[^a-z0-9\s]", " ", title)

    # Remove extra spaces
    title = re.sub(r"\s+", " ", title)

    return title.strip()


def clean_description(description):
    description = description.lower()

    # Remove URLs
    description = re.sub(r"https?://\S+", " ", description)

    # Replace punctuation with spaces
    description = re.sub(r"[^a-z0-9\s]", " ", description)

    # Remove extra spaces
    description = re.sub(r"\s+", " ", description)

    return description.strip()


df["clean_title"] = df["title"].apply(clean_title)

df["clean_description"] = df["description"].apply(clean_description)

# combine title x3 and description
df["clean_text"] = (
    df["clean_title"] + " "
    + df["clean_title"] + " "
    + df["clean_title"] + " "
    + df["clean_description"]
)


def parse_tags(tags):
    tag_list = tags.split("|")

    cleaned_tags = []

    for tag in tag_list:
        tag = tag.strip()

        if tag:
            cleaned_tags.append(tag)

    return cleaned_tags


df["tag_list"] = df["tags"].apply(parse_tags)

# remove rows with no tags
df = df[df["tag_list"].apply(len) > 0].copy()

print("Rows with usable tags before frequency filtering:", len(df))


# remove tags/genres with fewer than 100 mentions
MIN_TAG_COUNT = 100

tag_counts = {}

for tag_list in df["tag_list"]:

    # count each tag only once per novel
    for tag in set(tag_list):

        if tag not in tag_counts:
            tag_counts[tag] = 0

        tag_counts[tag] += 1


# keep only tags/genres that appear in at least 100 novels
allowed_tags = {
    tag
    for tag, count in tag_counts.items()
    if count >= MIN_TAG_COUNT
}


print("\n================================================")
print("TAG / GENRE FREQUENCY FILTER")
print("================================================")

print("Minimum mentions required:", MIN_TAG_COUNT)

print("Unique tags/genres before filtering:", len(tag_counts))

print("Unique tags/genres after filtering:", len(allowed_tags))

print("\nRemoved tags/genres:")

removed_tags = sorted(
    tag
    for tag, count in tag_counts.items()
    if count < MIN_TAG_COUNT
)

for tag in removed_tags:
    print(f"{tag:30} {tag_counts[tag]}")


# remove low-frequency tags from each novel
df["tag_list"] = df["tag_list"].apply(
    lambda tags: [
        tag
        for tag in tags
        if tag in allowed_tags
    ]
)


# remove rows with no tags after frequency filtering
df = df[
    df["tag_list"].apply(len) > 0
].copy()

print("\nRows after removing novels with no remaining tags:", len(df))


# use ALL tags/genres
# no main genre filtering
all_tags = sorted(
    allowed_tags
)

print("\n================================================")
print("ALL TAG / GENRE DATASET")
print("================================================")

print("Rows:", len(df))

print("\nNumber of unique tags/genres:", len(all_tags))

print("\nTags/genres:")
print(all_tags)

print("\nTag/genre distribution:")

tag_counts = {}

for tag in all_tags:

    count = df["tag_list"].apply(
        lambda x: tag in x
    ).sum()

    tag_counts[tag] = count

for tag, count in tag_counts.items():
    print(f"{tag:30} {count}")


# convert tags into a multi-label format
mlb = MultiLabelBinarizer(
    classes=all_tags
)

Y = mlb.fit_transform(df["tag_list"])

print("\nNumber of unique tags/genres:", len(mlb.classes_))

print("\nTags/genres:")
print(list(mlb.classes_))



# train / test split
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    Y,
    test_size=0.20,
    random_state=42
)

print("\nTraining examples:", len(X_train))
print("Testing examples:", len(X_test))

# tf-idf
vectorizer = TfidfVectorizer(
    ngram_range=(1, 3),
    min_df=2,
    max_features=100000,
    sublinear_tf=True
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF shape:")
print(X_train_tfidf.shape)



# train model
model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        class_weight="balanced"
    ),
    n_jobs=-1
)

print("\nTraining model...")

# learns the relationship between tf-idf values and labels
model.fit(X_train_tfidf, y_train)

print("Training complete!")



# evaluate
predictions = model.predict(X_test_tfidf)

print("\n========== MODEL EVALUATION ==========\n")

print(
    classification_report(
        y_test,
        predictions,
        target_names=mlb.classes_,
        zero_division=0
    )
)



# save model
model_data = {
    "model": model,
    "vectorizer": vectorizer,
    "mlb": mlb,
    "genres": all_tags,
    "min_tag_count": MIN_TAG_COUNT
}

joblib.dump(
    model_data,
    "webnovel_title_description_model.pkl"
)

print("\nModel saved as:")
print("webnovel_title_description_model.pkl")



# predict genres from a title and description
def predict_title(title, description, threshold=0.30):

    cleaned_title = clean_title(title)

    cleaned_description = clean_description(description)

    # combine title and description
    cleaned_text = (
        cleaned_title
        + " "
        + cleaned_description
    )

    # convert title + description to tf-idf
    X = vectorizer.transform([cleaned_text])

    # get probability for every tag
    probabilities = model.predict_proba(X)[0]

    # sort from highest probability to lowest
    results = sorted(
        zip(mlb.classes_, probabilities),
        key=lambda x: x[1],
        reverse=True
    )

    # keep tags above threshold
    predictions = [
        (tag, probability)
        for tag, probability in results
        if probability >= threshold
    ]

    return predictions


# demo
while True:

    print("\n================================================")
    print("DEMO")
    print("================================================")

    title = input("\nEnter a webnovel title (or type 'quit' to stop): ")

    if title.strip().lower() in ["quit", "exit", "q"]:
        print("\nStopping predictor...")
        break

    if title.strip() == "":
        print("\nNo title entered. Stopping predictor...")
        break

    description = input("\nEnter the webnovel description: ")

    results = predict_title(
        title,
        description
    )

    print("\n========== PREDICTIONS ==========\n")

    if len(results) == 0:
        print("No tags passed the threshold.")
        print("Try lowering the threshold.")
    else:
        for tag, probability in results:
            print(f"{tag:30} {probability:.1%}")

    print("\n================================================")
    print("Ready for another novel.")
    print("================================================")